# Spatial interaction through networks

- get pandarm network
- assign origins, destinations
  - origins ZSJ/grid, destinations POIs
  - grid based origin==destination
- isochrones
- number of accessible POIs
- travel time
- travel time matrix
  - conversion to Graph
  - Graph.build_travel_cost
- get r5py network
  - replicate above using multimodal data

In [ ]:
import datetime

import geopandas as gpd
import pandarm
import r5py

from libpysal import graph, kernels

In [ ]:
prague = gpd.read_file(
    "https://martinfleischmann.net/sds/spatial_graphs/data/zsj_prague_2021.gpkg"
)
prague = prague.set_index("NAZ_ZSJ")
prague.explore()

``` py
network = pandarm.Network.from_gdf(prague, network_type="walk")
network.save_hdf5("data/prague_walk.h5")
```

In [ ]:
network = pandarm.Network.from_hdf5("data/prague_walk.h5")

https://geoportalpraha.cz/data-a-sluzby/5d32db18a5d545a79ad3029917984a89

In [ ]:
schools = gpd.read_file('data/schools.zip')

In [ ]:
# temp fix
network.crs = schools.crs
network.edges_df = network.edges_df.set_crs(schools.crs, allow_override=True)
network.nodes_df = network.nodes_df.set_crs(schools.crs, allow_override=True)

In [ ]:
network.set_pois("schools", 2000, 5, schools.geometry.x, schools.geometry.y)

In [ ]:
nearest_schools = network.nearest_pois(2000, "schools", num_pois=5)

In [ ]:
nearest_schools

In [ ]:
network.nodes_df.plot(nearest_schools[1], markersize=.01, legend=True)

In [ ]:
art_school = schools.query("TYP_TXT == 'Základní umělecká škola'")
network.set_pois(category="art_school", maxdist=5000, maxitems=1, x_col=art_school.geometry.x, y_col=art_school.geometry.y)

In [ ]:
nearest_art_school = network.nearest_pois(distance=5000, category="art_school")

In [ ]:
ax = art_school.plot(markersize=1, color='red', zorder=2)
network.nodes_df.plot(nearest_art_school[1], markersize=.01, legend=True, ax=ax)
ax.set_axis_off()

Isochrones

In [ ]:
art_schoool_nodes = network.get_node_ids(art_school.geometry.x, art_school.geometry.y)
isochrones = network.nodes_in_range(art_schoool_nodes, radius=1000)

In [ ]:
isochrones

In [ ]:
art_schoool_nodes

In [ ]:
network.nodes_df.loc[isochrones["destination"]].dissolve(isochrones["source"].values).concave_hull(ratio=.2).explore()

In [ ]:
network.nodes_df.loc[isochrones["destination"]].explore(isochrones['length'].values)

Aggregation

In [ ]:
node_ids = network.get_node_ids(prague.centroid.x, prague.centroid.y)
network.set(node_ids, variable=prague["n_people"], name="population")

In [ ]:
pop_1km = network.aggregate(distance=1000, func="sum", decay="linear", name="population")

In [ ]:
pop_1km

In [ ]:
network.nodes_df.plot(pop_1km, markersize=.01, legend=True)

In [ ]:
pop_5km = network.aggregate(distance=5000, func="sum", decay="flat", name="population")
network.nodes_df.plot(pop_5km, markersize=.01, legend=True)

In [ ]:
pop_mean = network.aggregate(distance=2000, func="mean", decay="flat", name="population")
network.nodes_df.plot(pop_mean, markersize=.01, legend=True)

In [ ]:
# this needs new libpysal
# travel_cost_graph = graph.Graph.build_travel_cost(prague.set_geometry(prague.centroid), network=network, threshold=2000)
# travel_cost_graph

## r5py

https://download.geofabrik.de/europe/czech-republic.html

In [ ]:
transport_network = r5py.TransportNetwork(
    osm_pbf="data/praha-261020.osm.pbf",
    gtfs=["data/PID_GTFS.zip"],
    allow_errors=True,
)

Isochrones

In [ ]:
origins = art_school.copy()
origins['id'] = art_school.NAZEV_ZAR

isochrones = r5py.Isochrones(
    transport_network,
    origins=origins,
    departure=datetime.datetime(2026, 10, 22, 10, 30),
    transport_modes=[r5py.TransportMode.TRANSIT, r5py.TransportMode.WALK],
    isochrones=[5, 10, 15],
)

In [ ]:
isochrones['travel_time'] = isochrones['travel_time'].astype(int)
isochrones.plot('travel_time', legend=True)

TT matrix

In [ ]:
destinations = prague.loc[["Albertov"]].set_geometry(prague.centroid)
origins = prague.set_geometry(prague.centroid)

destinations["id"] = destinations.index
origins["id"] = origins.index

travel_times = r5py.TravelTimeMatrix(
    transport_network,
    origins=origins,
    destinations=destinations,
    departure=datetime.datetime(2026, 10, 22, 10, 30),
    transport_modes=[
        r5py.TransportMode.TRANSIT,
        r5py.TransportMode.WALK,
    ],
    snap_to_network=True,
)
travel_times

In [ ]:
prague.plot(travel_times.travel_time.values, legend=True)

Full matrix

In [ ]:
destinations = prague.set_geometry(prague.centroid)
origins = destinations

destinations["id"] = destinations.index

travel_times = r5py.TravelTimeMatrix(
    transport_network,
    origins=origins,
    destinations=destinations,
    departure=datetime.datetime(2026, 10, 22, 10, 30),
    transport_modes=[
        r5py.TransportMode.TRANSIT,
        r5py.TransportMode.WALK,
    ],
    snap_to_network=True,
    max_time=datetime.timedelta(hours=1),
)
travel_times

In [ ]:
transit_graph = graph.Graph.from_adjacency(
    adjacency=travel_times.dropna(),
    focal_col="from_id",
    neighbor_col="to_id",
    weight_col="travel_time",
)

In [ ]:
transit_graph["Albertov"]

In [ ]:
travel_times["weights"] = kernels.kernel(travel_times["travel_time"], bandwidth=60, kernel="bisquare")
travel_times

In [ ]:
transit_graph_weighted = graph.Graph.from_adjacency(
    adjacency=travel_times.dropna(),
    focal_col="from_id",
    neighbor_col="to_id",
    weight_col="weights",
)

In [ ]:
transit_graph_weighted["Albertov"]

In [ ]:
prague.assign(w=transit_graph_weighted["Albertov"]).plot('w', legend=True)